# 07 — Similarity, Anomaly & Synthesis

The finale: turn the feature store into GOAT-Life-usable tools and consolidate.

1. **Lookalike engine** — cosine k-NN on the feature + embedding space ("find localities like X", incl.
   cross-city lookalike expansion)
2. **Anomaly detection** (IsolationForest) — unusual localities + **hidden gems** (attractive but under-priced/under-covered)
3. **ICP / Locality Attractiveness Score** — a magicbricks-side composite from imputed affluence + corporate/
   sector + youth + access + centrality, with a GO / SAMPLE-FIRST / WAIT verdict
4. **Consolidate** → `artifacts/localities_features_master.parquet` + a committed `INSIGHTS.md`

> **Scope note:** these notebooks use magicbricks data only. The production "Where to Win" pipeline layers
> **gym-fitness density** and **darkstore serviceability** on top — those are not recomputed here, so this
> score is the *demand-context* half of the full GOAT-Fit, not a replacement for it.

Reads `artifacts/features_imputed.parquet` + `artifacts/embeddings.npz`.

In [1]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

ART = Path.cwd() / "artifacts"
df = pd.read_parquet(ART / "features_imputed.parquet").reset_index(drop=True)
emb = np.load(ART / "embeddings.npz", allow_pickle=True)["combined"]
print("Rows:", len(df), "| cols:", df.shape[1])

Rows: 1001 | cols: 118


In [2]:
# --- Build the similarity/anomaly matrix (complete, scaled features + embedding PCA) ---
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA

SIM = ["res_avg_buy_imputed", "total_amenities", "amenity_diversity", "infra_completeness",
       "num_employers", "primary_sector_score", "sector_information_technology",
       "sector_manufacturing_and_industry", "sector_government_and_public_sector",
       "num_educational_institute", "num_hospital", "is_metro_connected",
       "dist_to_city_centroid_km", "knn_density_km", "graph_degree", "graph_pagerank", "belt_size"]
SIM = [c for c in SIM if c in df.columns]
Xt = df[SIM].apply(pd.to_numeric, errors="coerce")
Xt_s = StandardScaler().fit_transform(SimpleImputer(strategy="median").fit_transform(Xt))
emb_pca = PCA(n_components=20, random_state=42).fit_transform(StandardScaler().fit_transform(emb))
M = np.hstack([Xt_s, emb_pca])
print("Similarity matrix:", M.shape)

Similarity matrix: (1001, 37)


In [3]:
# --- Lookalike engine (cosine k-NN) ---
from sklearn.neighbors import NearestNeighbors

nn = NearestNeighbors(n_neighbors=11, metric="cosine").fit(M)
dist, idx = nn.kneighbors(M)

def lookalikes(area_query, k=5, cross_city=False):
    matches = df.index[df["AREA"].str.contains(area_query, case=False, na=False)]
    if len(matches) == 0:
        return f"No locality matching '{area_query}'"
    i = matches[0]
    out = []
    for j, d in zip(idx[i][1:], dist[i][1:]):
        if cross_city and df.at[j, "ADDRESS"] == df.at[i, "ADDRESS"]:
            continue
        out.append((df.at[j, "AREA"], df.at[j, "ADDRESS"], round(1 - d, 3)))
        if len(out) >= k:
            break
    return df.at[i, "AREA"], out

# store each locality's nearest cross-city lookalike
nearest_cc = []
for i in range(len(df)):
    pick = None
    for j in idx[i][1:]:
        if df.at[j, "ADDRESS"] != df.at[i, "ADDRESS"]:
            pick = df.at[j, "AREA"] + " (" + str(df.at[j, "ADDRESS"]) + ")"
            break
    nearest_cc.append(pick)
df["nearest_lookalike_cross_city"] = nearest_cc

q = lookalikes("Golf Course Road", k=5, cross_city=True)
print("Cross-city lookalikes for:", q[0])
for a, c, s in q[1]:
    print(f"  {a} ({c})  sim={s}")

Cross-city lookalikes for: Golf Course Road, Gurgaon


In [4]:
# --- Anomaly detection (IsolationForest) ---
from sklearn.ensemble import IsolationForest

iso = IsolationForest(n_estimators=300, contamination=0.06, random_state=42).fit(M)
df["anomaly_score"] = iso.decision_function(M).round(4)   # lower = more anomalous
df["is_anomaly"] = (iso.predict(M) == -1)
print("Flagged anomalies:", int(df["is_anomaly"].sum()))
print("Most unusual localities:")
print(df.nsmallest(6, "anomaly_score")[["AREA", "ADDRESS", "res_avg_buy_imputed"]].to_string(index=False))

Flagged anomalies: 60
Most unusual localities:
                   AREA    ADDRESS  res_avg_buy_imputed
         Hadapsar, Pune       Pune               7950.0
    Bandra East, Mumbai     Mumbai              43950.0
      Kothur, Hyderabad  Hyderabad               6313.0
   Bhowanipore, Kolkata    Kolkata              10200.0
Vasant Vihar, New Delhi  New Delhi              30650.0
  Sector 22, Chandigarh Chandigarh              21482.0


In [5]:
# --- ICP / Locality Attractiveness Score (magicbricks-side composite) + verdict ---
def pct(s):
    return s.rank(pct=True) * 100

affluence = pct(df["res_avg_buy_imputed"])
# corporate now uses the quality-weighted employer score (Improvement 2 downstream) + sector mix
corporate = pct(df["employer_quality"].fillna(0)
                + 5 * df[["sector_information_technology", "sector_finance_and_banking", "sector_consulting"]].fillna(0).sum(axis=1))
youth = pct(df["num_educational_institute"].fillna(0))
access = pct(df["is_metro_connected"].astype(float) + df["infra_completeness"].fillna(0))
centrality = pct(df["graph_pagerank"].fillna(0))

# (a) hand-weighted ICP
df["icp_score"] = (0.30 * affluence + 0.30 * corporate + 0.10 * youth
                   + 0.15 * access + 0.15 * centrality).round(1)
df["icp_verdict"] = pd.cut(df["icp_score"], [-1, 45, 70, 101], labels=["WAIT", "SAMPLE-FIRST", "GO"])

# (b) Improvement 9a: PCA weights = direction of maximum VARIANCE across the 5 components
# (unsupervised). This is NOT calibration against an outcome (none exists) -> labelled
# "variance-implied", not "learned". Sign of PC1 is arbitrary, so we orient it positive.
from sklearn.decomposition import PCA
comp = np.column_stack([affluence, corporate, youth, access, centrality])
w = PCA(n_components=1).fit(comp).components_[0]
if w.sum() < 0:
    w = -w
w = w / np.abs(w).sum()
print("Variance-implied ICP weights:",
      dict(zip(["affluence", "corporate", "youth", "access", "centrality"], w.round(3))))
df["icp_score_pca"] = (comp * w).sum(axis=1).round(1)

# (c) Improvement 9b: Pareto frontier - weight-free dominance across all 5 dimensions
def pareto_mask(costs):
    opt = np.ones(costs.shape[0], dtype=bool)
    for i, c in enumerate(costs):
        if opt[i]:
            opt[i] = not np.any(np.all(costs >= c, axis=1) & np.any(costs > c, axis=1))
    return opt
df["pareto_optimal"] = pareto_mask(comp)

# hidden gems: (i) ICP-based value play + (ii) Improvement 10 graph-spillover (orthogonal)
median_price = df["res_avg_buy_imputed"].median()
df["hidden_gem"] = (df["icp_score"] >= 65) & ((df["res_avg_buy_imputed"] < median_price) | df["price_is_imputed"])
df["spillover_gem"] = (df["price_gap_pct"] > 0.20) & (df["graph_degree"] >= 2) & df["neighbor_avg_price"].notna()
df["hidden_gem_v2"] = df["hidden_gem"] | df["spillover_gem"]

print("ICP verdict distribution:", df["icp_verdict"].value_counts().to_dict())
print("Pareto-optimal localities:", int(df["pareto_optimal"].sum()))
print("Hidden gems:", int(df["hidden_gem"].sum()), "| spillover gems:", int(df["spillover_gem"].sum()),
      "| combined v2:", int(df["hidden_gem_v2"].sum()))
print()
print("Top 8 ICP localities:")
print(df.nlargest(8, "icp_score")[["AREA", "ADDRESS", "icp_score", "icp_verdict"]].to_string(index=False))

Variance-implied ICP weights: {'affluence': np.float64(-0.039), 'corporate': np.float64(0.253), 'youth': np.float64(0.258), 'access': np.float64(0.292), 'centrality': np.float64(0.158)}
ICP verdict distribution: {'SAMPLE-FIRST': 509, 'WAIT': 378, 'GO': 114}
Pareto-optimal localities: 84
Hidden gems: 70 | spillover gems: 38 | combined v2: 105

Top 8 ICP localities:
                    AREA   ADDRESS  icp_score icp_verdict
    Sushant Lok, Gurgaon  Gurugram       88.5          GO
        Saket, New Delhi New Delhi       88.0          GO
      Sector 43, Gurgaon  Gurugram       87.0          GO
    Phase 2 DLF, Gurgaon  Gurugram       84.9          GO
           Powai, Mumbai    Mumbai       84.1          GO
Banjara Hills, Hyderabad Hyderabad       84.0          GO
      Sector 47, Gurgaon  Gurugram       83.8          GO
    Andheri West, Mumbai    Mumbai       83.6          GO


In [6]:
# --- Consolidate: master feature store + INSIGHTS.md ---
master = ART / "localities_features_master.parquet"
df.to_parquet(master, index=False)

def md_table(frame, cols):
    head = "| " + " | ".join(cols) + " |\n| " + " | ".join("---" for _ in cols) + " |\n"
    rows = "".join("| " + " | ".join(str(r[c]) for c in cols) + " |\n" for _, r in frame.iterrows())
    return head + rows

lines = []
lines.append("# GOAT Life — Locality ML Insights\n")
lines.append(f"_Generated from {len(df)} localities across {df['ADDRESS'].nunique()} cities "
             f"(magicbricks ML pipeline, notebooks 01-07)._\n")
lines.append("\n## Top 15 ICP localities (demand-context attractiveness)\n")
lines.append(md_table(df.nlargest(15, "icp_score")[["AREA", "ADDRESS", "icp_score", "icp_verdict", "archetype_ml"]]
                      .round({"icp_score": 1}),
                      ["AREA", "ADDRESS", "icp_score", "icp_verdict", "archetype_ml"]))
lines.append(f"\n## Hidden gems ({int(df['hidden_gem'].sum())}) — attractive but under-priced / under-covered\n")
hg = df[df["hidden_gem"]].nlargest(12, "icp_score")
lines.append(md_table(hg[["AREA", "ADDRESS", "icp_score", "res_avg_buy_imputed", "price_is_imputed"]]
                      .round({"icp_score": 1, "res_avg_buy_imputed": 0}),
                      ["AREA", "ADDRESS", "icp_score", "res_avg_buy_imputed", "price_is_imputed"]))
lines.append(f"\n## Spillover gems ({int(df['spillover_gem'].sum())}) - cheaper than graph neighbours (spatial arbitrage)\n")
sg = df[df["spillover_gem"]].nlargest(10, "price_gap_pct")
lines.append(md_table(sg[["AREA", "ADDRESS", "price_gap_pct", "res_avg_buy", "neighbor_avg_price"]]
                      .round({"price_gap_pct": 2, "res_avg_buy": 0, "neighbor_avg_price": 0}),
                      ["AREA", "ADDRESS", "price_gap_pct", "res_avg_buy", "neighbor_avg_price"]))
lines.append("\n## Top affluence drivers (SHAP, from NB06)\n")
shap_csv = ART / "shap_drivers.csv"
if shap_csv.exists():
    sd = pd.read_csv(shap_csv, index_col=0).head(10)
    lines.append("| Feature | Mean abs(SHAP) |\n| --- | --- |\n")
    for feat, row in sd.iterrows():
        lines.append(f"| {feat} | {row.iloc[0]:.1f} |\n")
lines.append("\n## Data-driven archetypes (NB05)\n")
av = df["archetype_ml"].value_counts()
lines.append(md_table(av.reset_index().rename(columns={"index": "archetype", "archetype_ml": "n"})
                      if "index" in av.reset_index().columns else
                      av.rename_axis("archetype").reset_index(name="n"),
                      ["archetype", "n"]))
lines.append("\n## Largest belts (NB04 — contiguous groups to attack together)\n")
belt = df[df["belt_size"] > 1].groupby("belt_id").agg(
    city=("ADDRESS", "first"), size=("belt_size", "first")).nlargest(8, "size")
lines.append(md_table(belt.reset_index(), ["belt_id", "city", "size"]))
lines.append("\n## Method & caveats\n")
lines.append("- ICP = 0.30 affluence + 0.30 corporate + 0.10 youth + 0.15 access + 0.15 centrality "
             "(percentile-ranked). Magicbricks-side only.\n")
lines.append("- Prices for ~45% of localities are **model-imputed** (LightGBM, CV R2 0.52, beats city-mean) "
             "and flagged `price_is_imputed` — treat as directional.\n")
lines.append("- Production GOAT-Fit additionally layers gym-fitness density + darkstore serviceability "
             "(not in these notebooks).\n")

ins = Path.cwd() / "INSIGHTS.md"
ins.write_text("".join(lines), encoding="utf-8")
print("Saved master parquet:", df.shape, "->", master.relative_to(Path.cwd()))
print("Saved insights ->", ins.name)

Saved master parquet: (1001, 128) -> artifacts\localities_features_master.parquet
Saved insights -> INSIGHTS.md


## Outputs
- `artifacts/localities_features_master.parquet` — the full consolidated feature store (the artifact the
  production pipeline can ingest), now with lookalikes, anomaly flags, ICP score/verdict, and hidden-gem flags.
- `INSIGHTS.md` — top ICP localities, hidden gems, archetypes, and belts.

**Pipeline complete (NB01-07).** From 1,001 raw localities × 16 columns, the ML pipeline derived a rich,
documented feature store and a ranked, explainable view of where GOAT Life's demand-context is strongest.